In [1]:
from scipy.optimize import brentq
import numpy as np

In [2]:
def E_anode(cV2, cV3, E0_a, R, T, F):
    return E0_a + (R*T/F) * np.log(cV3 / cV2)

def E_cathode(cVO2, cVO2p, cH, E0_c, R, T, F):
    return E0_c + (R*T/F) * np.log((cVO2 * cH**2) / cVO2p)

def E_cell(Ec, Ea, cH_pos, cH_neg, R, T, F):
    return Ec - Ea + (R*T/F) * np.log(cH_pos / cH_neg)

In [3]:
def i0_anode(F, a, k_a, cV3, cV2, alpha_a):
    return F * a * k_a * (cV3**alpha_a) * (cV2**(1 - alpha_a))

def i0_cathode(F, a, k_c, cVO2p, cVO2, alpha_c):
    return F * a * k_c * (cVO2p**alpha_c) * (cVO2**(1 - alpha_c))

In [4]:
def i_bv(i0, eta, alpha, R, T, F):
    term1 = np.exp((1 - alpha) * F * eta / (R * T))
    term2 = np.exp(-alpha * F * eta / (R * T))
    return i0 * (term1 - term2)

In [ ]:
def current_density(c, params):
    R, T, F = params["R"], params["T"], params["F"]
    # Potenciales de equilibrio
    Ea = E_anode(c["V2+"], c["V3+"], params["E0_a"], R, T, F)
    Ec = E_cathode(c["VO2+"], c["VO2"], c["H+"], params["E0_c"], R, T, F)
    # Donnan
    if params.get("use_donnan", False):
        Erev = E_cell(Ec, Ea, c["H+_pos"], c["H+_neg"], R, T, F)
    else:
        Erev = Ec - Ea

    A_cell = params["A_cell"]
    R_ohm  = params["R_ohm"]

    def f_i(i):
        eta_tot = params["Vcell"] - Erev - (i * A_cell) * R_ohm
        eta_c = params["beta"] * eta_tot
        eta_a = -(1 - params["beta"]) * eta_tot
        ic = i_bv(i0_cathode(F, params["a"], params["k_c"],
                             c["VO2+"], c["VO2"], params["alpha_c"]),
                  eta_c, params["alpha_c"], R, T, F)
        ia = i_bv(i0_anode(F, params["a"], params["k_a"],
                           c["V3+"], c["V2+"], params["alpha_a"]),
                  eta_a, params["alpha_a"], R, T, F)
        return ic + ia  # debe igualar 0

### REVISAR

        # --- Solución para brentq ---
        
    lo, hi = -5e3, 5e3  # intervalo inicial más razonable (A/m²)
    f_lo, f_hi = f_i(lo), f_i(hi)

    # Si no hay cambio de signo, expandimos el rango gradualmente
    expansion = 1
    while np.sign(f_lo) == np.sign(f_hi) and expansion < 50:
        lo *= 1.5
        hi *= 1.5
        f_lo, f_hi = f_i(lo), f_i(hi)
        expansion += 1

    if np.sign(f_lo) == np.sign(f_hi):
        raise RuntimeError("No se encontró cambio de signo en f_i dentro del rango ampliado.")

    # Resolver raíz
    i_solution = brentq(f_i, lo, hi, maxiter=200, xtol=1e-9)
    return i_solution
